# Stanford Flow measurements

### loading the old pickle file where i saved the autoflow and manual flow measurements that were done originally. dumping this as a csv so that i am not locked into old conda environment and can instead use the new auto-flow_3-9 environment

#### imports

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
from numpy.linalg import inv
import math
import random
import sys
import imageio
import os

import scipy
from scipy.ndimage import zoom, laplace, gaussian_filter, rotate
from scipy import ndimage
from scipy.interpolate import RegularGridInterpolator as RGI
from scipy.interpolate import splprep, splev
from skimage import measure
from skimage.morphology import skeletonize_3d
from skimage.filters import threshold_otsu
from skimage.morphology import skeletonize
from skimage import exposure
print("python: ", sys.version)
print("scipy: ", scipy.__version__)

import matplotlib.pyplot as plt
from matplotlib.path import Path
from ipywidgets import interact, IntSlider, Layout
# import ipyvolume as ipv

import pickle
import csv
import h5py
import pydicom
import json

import os
from tqdm import tqdm
import traceback
import datetime
import copy
import gc
import time
from tqdm import tqdm
import multiprocessing as mp

# Test if gdcm works for decompressing JPEG 2000
print("IT IS {}, GDCM IS LOCKED AND LOADED".format(pydicom.pixel_data_handlers.gdcm_handler.is_available()))

# import cv2
# from mpl_toolkits.mplot3d import Axes3D
# import matplotlib
# matplotlib.use('Agg')

python:  3.7.13 (default, Mar 29 2022, 02:18:16) 
[GCC 7.5.0]
scipy:  1.7.3
IT IS True, GDCM IS LOCKED AND LOADED


#### versions

In [4]:
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import scipy
import imageio
import h5py
import pydicom
import matplotlib
import ipywidgets
import multiprocessing
import skimage

# Optional imports, uncomment if needed and installed
# import ipyvolume as ipv
# import cv2

def check_package_versions():
    print("Python version:", sys.version)
    print("NumPy version:", np.__version__)
    print("Pandas version:", pd.__version__)
    print("Seaborn version:", sns.__version__)
    print("SciPy version:", scipy.__version__)
    print("ImageIO version:", imageio.__version__)
    print("H5py version:", h5py.__version__)
    print("Pydicom version:", pydicom.__version__)
    print("Matplotlib version:", matplotlib.__version__)
    print("IPyWidgets version:", ipywidgets.__version__)
    print("Multiprocessing available:", multiprocessing.cpu_count(), "cores")
    print("Scikit-Image version:", skimage.__version__)

    # Uncomment if ipyvolume is installed and import is uncommented
    # print("IPyVolume version:", ipv.__version__)

    # Uncomment if OpenCV is installed and import is uncommented
    # print("OpenCV version:", cv2.__version__)

    # Testing GDCM availability in Pydicom for handling DICOM files
    try:
        gdcm_loaded = pydicom.pixel_data_handlers.gdcm_handler.is_available()
        print("GDCM is available for Pydicom:", gdcm_loaded)
    except Exception as e:
        print("GDCM check failed:", e)

# Call the function to print versions
check_package_versions()


Python version: 3.7.13 (default, Mar 29 2022, 02:18:16) 
[GCC 7.5.0]
NumPy version: 1.21.5
Pandas version: 1.3.5
Seaborn version: 0.11.2
SciPy version: 1.7.3
ImageIO version: 2.9.0
H5py version: 2.10.0
Pydicom version: 2.3.0
Matplotlib version: 3.5.1
IPyWidgets version: 8.1.1
Multiprocessing available: 16 cores
Scikit-Image version: 0.19.2
GDCM is available for Pydicom: True


#### load pickle file

In [5]:
DUMPFOLDER = '/maxwell-projects/Aorta_pulmonary_artery_localization/testing_compiled_volumes'
DUMPPATH = os.path.join(DUMPFOLDER,'testing_gifs_segnet-ak_spline_AV-midAo_computations_ah-pr-ls.pkl')

df_og = pd.read_pickle(DUMPPATH)
display(df_og)



,AV,Ao,PV,PA,Qp/Qs,A1_auto,A2_auto,A3_auto,A4_auto,A5_auto,...,AV_avg,Ao_avg,PV_avg,PA_avg,Qp/Qs_avg,AV_LS,Ao_LS,PV_LS,PA_LS,Qp/Qs_LS
Phonetic,,,,,,,,,,,,,,,,,,,,,
Besapol,4.26,4.34,4.54,5.70,1.190698,4.324760,4.236766,4.381345,4.184108,3.884214,...,4.713333,4.390000,4.920000,5.440000,1.239180,4.30,4.60,5.01,4.99,1.123596
Deefegi,4.78,4.38,12.04,12.83,2.715066,4.455792,4.573752,5.021597,5.016327,5.063245,...,4.713333,4.350000,12.026667,13.163333,3.026054,4.70,4.00,12.03,13.39,2.921839
Dehithu,6.99,6.37,7.62,7.53,1.133982,8.125992,8.100068,6.723978,6.870266,6.509243,...,6.786667,6.150000,7.410000,7.910000,1.286179,6.97,5.71,6.76,8.82,1.228707
Dukasa,4.81,4.67,9.20,9.73,1.996835,4.186430,4.350754,4.613100,4.665200,3.876967,...,4.876667,4.596667,9.216667,9.820000,2.136331,4.90,4.50,9.70,10.00,2.095745
Fegaygun,4.90,4.77,11.29,11.38,2.344364,4.029989,4.122319,4.263593,4.584373,4.788201,...,4.903333,4.846667,11.343333,11.270000,2.325309,4.80,5.00,11.30,11.30,2.306122
Fenoopub,6.00,5.11,5.63,6.38,1.081008,6.158968,5.948341,5.679290,5.478890,5.109586,...,6.103333,5.256667,5.666667,6.456667,1.228282,6.20,5.40,5.60,6.60,1.051724
Fratalooy,5.04,6.09,5.32,6.33,1.046721,5.370384,5.385551,6.152368,5.777844,4.322579,...,5.176667,5.916667,4.896667,6.043333,1.021408,4.94,5.80,3.65,5.14,0.818436
Gajefe,5.60,6.75,10.04,10.06,1.627530,5.516221,6.180746,6.236769,6.044409,6.236464,...,6.003333,6.806667,10.163333,10.150000,1.491185,6.20,6.80,10.80,10.40,1.630769
Giretap,2.69,2.44,2.50,2.20,0.916179,2.625707,2.703844,2.792115,2.931085,2.973513,...,2.556667,2.253333,2.163333,1.706667,0.757396,2.30,2.10,1.85,0.97,0.640909


#### dump file as csv

In [6]:
## save as a csv
df_og.to_csv(os.path.join(DUMPFOLDER,'flow_measurements_og.csv'))

GE_TESTING_BASE_PATH = '/maxwell-projects/Aorta_pulmonary_artery_localization/ge_testing/'
df_og.to_csv(os.path.join(GE_TESTING_BASE_PATH,'flow_measurements_og.csv'))